# 1. Data Validation and Initial Analysis

## Objective
The objective of this notebook is to validate the processed Online Retail II
dataset before performing Exploratory Data Analysis (EDA).

The validation process checks:
- Whether the processed files exist.
- Whether the expected columns are available.
- Whether missing values are present.
- Whether numeric variables contain valid values.
- Whether the dataset is suitable for further analysis.

The processed dataset will later be used to investigate the relationship
between product price and quantity sold.

In [5]:
# Import libraries required for data validation and analysis.
import pandas as pd
import numpy as np
import json

from pathlib import Path
from IPython.display import display

# Display more columns and format numbers for readability.
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Identify the project root directory.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Define paths for processed data and the audit report.
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
SALES_FILE = PROCESSED_DIR / "online_retail_sales_initial.csv"
AUDIT_FILE = PROCESSED_DIR / "processing_audit.json"

print("Project root:", PROJECT_ROOT)
print("Processed data file exists:", SALES_FILE.exists())
print("Audit file exists:", AUDIT_FILE.exists())

Project root: c:\intern_project\econocausal-dml
Processed data file exists: True
Audit file exists: True


### Analysis

This cell verifies that the required processed dataset and processing audit
are available.

If both files exist, the next step is to inspect the processing summary
and validate the structure of the saved data.

If a file is missing, the file path or data-processing step must be checked
before continuing.

In [7]:
# Load the summary generated during data processing.
with open(AUDIT_FILE, "r", encoding="utf-8") as file:
    audit = json.load(file)

# Display the audit as a readable table.
audit_df = pd.DataFrame(
    list(audit.items()),
    columns=["Metric", "Value"]
)

display(audit_df)

,Metric,Value
0,source_file,online_retail_II.xlsx
1,sheets_processed,"[online_retail_2009_2010.csv, online_retail_20..."
2,rows_read,1067371
3,rows_missing_customer_id,243007
4,rows_missing_description,4382
5,exact_duplicate_rows,12131
6,cancellation_invoice_rows,19494
7,negative_quantity_rows,22950
8,zero_price_rows,6202
9,negative_price_rows,5


### Analysis of Processing Audit

The processing audit summarizes the number of rows read, missing values,
cancellation invoices, negative quantities, and price-related issues.

These metrics help identify potential data-quality concerns.

Missing customer IDs do not necessarily mean that a transaction is invalid.
Similarly, negative quantities may represent product returns.

The audit's duplicate count was calculated within individual processing
chunks. Therefore, it should not be interpreted as a complete global
duplicate count.

In [8]:
# Load a sample to validate the processed CSV without loading it all again.
df = pd.read_csv(
    SALES_FILE,
    nrows=10000,
    parse_dates=["invoicedate"],
    low_memory=False
)

print("Sample dimensions:", df.shape)
print("Number of columns:", len(df.columns))

display(df.head())

Sample dimensions: (10000, 9)
Number of columns: 9


,invoice,stockcode,description,quantity,invoicedate,unit_price,customer_id,country,line_sales_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,30.00


### Analysis of Dataset Structure

The sample contains transaction-level records from the Online Retail II
dataset.

Important variables include:

- `invoice`: Transaction identifier.
- `stockcode`: Product identifier.
- `invoicedate`: Date and time of the transaction.
- `quantity`: Number of units purchased in a transaction line.
- `unit_price`: Recorded unit price of the product.
- `customer_id`: Customer identifier, which may be missing.
- `country`: Customer's country.
- `line_sales_value`: Quantity multiplied by unit price.

These variables provide the foundation for examining price variation and
purchasing patterns. However, transaction-level records alone do not
establish the causal effect of price on demand.

In [9]:
# Calculate missing counts and percentages for the sample.
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100
}).sort_values("Missing Percentage", ascending=False)

display(missing_summary)

,Missing Count,Missing Percentage
customer_id,2978,29.78
invoice,0,0.00
stockcode,0,0.00
quantity,0,0.00
description,0,0.00
invoicedate,0,0.00
unit_price,0,0.00
country,0,0.00
line_sales_value,0,0.00


### Analysis of Missing Values

Missing values must be evaluated according to the purpose of each variable.

Customer IDs may be missing even when the transaction contains a valid
product, quantity, and price. Such records can still contribute to
transaction-level descriptive analysis.

Missing product descriptions may be investigated using the product code.
Rows should not be removed automatically without understanding the reason
for missingness.

For causal modeling, missing customer information may limit our ability
to control for customer-specific purchasing behavior.

In [10]:
# Inspect the distribution of quantity, unit price, and line sales value.
numeric_columns = [
    "quantity",
    "unit_price",
    "line_sales_value"
]

display(df[numeric_columns].describe().T)

# Check basic conditions expected in this initial positive-sales sample.
print("All quantities positive:", df["quantity"].gt(0).all())
print("All unit prices positive:", df["unit_price"].gt(0).all())
print("All sales values non-missing:",
      df["line_sales_value"].notna().all())

# Confirm that the sales value agrees with quantity multiplied by price.
expected_value = df["quantity"] * df["unit_price"]

difference = (
    df["line_sales_value"] - expected_value
).abs()

print(
    "Maximum sales-value calculation difference:",
    difference.max()
)

,count,mean,std,min,25%,50%,75%,max
quantity,"10,000.00",11.17,93.85,1.00,1.00,3.00,8.00,"5,184.00"
unit_price,"10,000.00",4.26,17.49,0.06,1.25,2.57,4.95,922.05
line_sales_value,"10,000.00",20.32,74.43,0.19,3.90,9.12,17.40,"1,800.00"


All quantities positive: True
All unit prices positive: True
All sales values non-missing: True
Maximum sales-value calculation difference: 5.684341886080802e-14


### Analysis of Numeric Variables

The descriptive statistics summarize the typical values and variability
of quantity, unit price, and line sales value.

The mean and standard deviation help describe the distribution, while
the median and upper percentiles help identify potential skewness.

Large quantities or prices should be investigated rather than automatically
removed. They may represent genuine bulk purchases or unusual transactions.

Positive quantities and prices are expected because the initial sales
dataset was created using those filters. This check validates the saved
sample but does not prove that every record is free of errors.

In [11]:
# Examine the time coverage and product/invoice identifiers.
print("Earliest transaction:", df["invoicedate"].min())
print("Latest transaction:", df["invoicedate"].max())

print("Unique products in sample:", df["stockcode"].nunique())
print("Unique invoices in sample:", df["invoice"].nunique())
print("Unique countries in sample:", df["country"].nunique())

Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2009-12-04 09:43:00
Unique products in sample: 2129
Unique invoices in sample: 365
Unique countries in sample: 12


## Initial Validation Conclusion

The processed sample has been inspected for structure, missing values,
numeric consistency, and date coverage.

This stage establishes whether the saved data can be loaded and used
for further descriptive analysis.

The next stage will perform full Exploratory Data Analysis, including:

1. Distribution of product prices and quantities.
2. Detection and investigation of outliers.
3. Monthly sales and price trends.
4. Product-level price variation.
5. Descriptive relationships between price and quantity.

Causal estimation will be considered only after defining the analysis unit,
treatment, outcome, potential confounders, and limitations of the available
transaction data.